## GPU measurement

BBP and physical compression at round 300. Weak, Balanced and KuaiRec; five paired seeds per environment.


In [1]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import t as student_t

In [2]:
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
if not (ROOT / 'scripts/gpu.py').is_file():
    ROOT = next(Path('../input').rglob('scripts/gpu.py')).parents[1]
INPUTS = ROOT / 'experiments/GPU/inputs'
OUTPUT = Path('gpu_results')
METHODS = ('BBP', 'BBP + physical compression')

### Runs

Each run uses a fresh process. Warm-up precedes timing.


In [3]:
(OUTPUT / 'runs').mkdir(parents=True, exist_ok=False)
plan = json.loads((INPUTS / 'protocol.json').read_text())
records = []

In [4]:
for index, job in enumerate(plan['jobs']):
    order = METHODS if index % 5 % 2 == 0 else METHODS[::-1]
    for method in order:
        print(f"{len(records) + 1}/30  {job['case']}  seed {job['seed']}  {method}", flush=True)
        subprocess.run([sys.executable, '-m', 'scripts.gpu',
                        str(INPUTS.resolve()), str(OUTPUT.resolve()), str(index), method],
                       check=True, cwd=ROOT)
        suffix = 'BBP' if method == 'BBP' else 'compressed'
        record = json.loads((OUTPUT / 'runs' / f"{job['case']}_{job['seed']}_{suffix}.json").read_text())
        records.append(record)
        print(f"  {record['timing']['wall_seconds']:.1f} s; regret {record['summary']['regret']:.3f}; "
              f"{record['parameters']['final']['total']} parameters", flush=True)

1/30  weak  seed 8300  BBP
  67.0 s; regret 403.899; 17591 parameters
2/30  weak  seed 8300  BBP + physical compression
  64.6 s; regret 399.241; 3457 parameters
3/30  weak  seed 8301  BBP + physical compression
  63.0 s; regret 379.202; 3575 parameters
4/30  weak  seed 8301  BBP
  63.5 s; regret 401.546; 17591 parameters
5/30  weak  seed 8302  BBP
  62.4 s; regret 288.908; 17591 parameters
6/30  weak  seed 8302  BBP + physical compression
  63.2 s; regret 285.927; 3575 parameters
7/30  weak  seed 8303  BBP + physical compression
  64.1 s; regret 329.621; 3341 parameters
8/30  weak  seed 8303  BBP
  63.8 s; regret 326.829; 17591 parameters
9/30  weak  seed 8304  BBP
  63.9 s; regret 345.910; 17591 parameters
10/30  weak  seed 8304  BBP + physical compression
  63.8 s; regret 348.852; 3695 parameters
11/30  balanced  seed 8300  BBP
  64.3 s; regret 506.555; 17591 parameters
12/30  balanced  seed 8300  BBP + physical compression
  63.6 s; regret 506.885; 3695 parameters
13/30  balanced  

### Results

Ratios are compressed / BBP, with 95% paired intervals in log space.


In [5]:
hardware = records[0]['hardware']
(OUTPUT / 'hardware.json').write_text(json.dumps(hardware, indent=2, allow_nan=False) + '\n')
runs = pd.DataFrame([dict(Environment=r['case'], Seed=r['seed'], Method=r['method'],
                         Wall_s=r['timing']['wall_seconds'],
                         Total_parameters=r['parameters']['final']['total'],
                         Regret=r['summary']['regret']) for r in records])


In [6]:
table = []

for case, title in (('weak', 'Weak'), ('balanced', 'Balanced'), ('kuairec', 'KuaiRec')):
    a = runs[(runs.Environment == case) & (runs.Method == METHODS[0])].sort_values('Seed')
    b = runs[(runs.Environment == case) & (runs.Method == METHODS[1])].sort_values('Seed')
    if len(a) != 5 or len(b) != 5 or a.Seed.tolist() != b.Seed.tolist():
        raise ValueError('Five paired seeds per environment are required.')
    row = dict(Environment=title, BBP_time_s=float(a.Wall_s.mean()),
               Compressed_time_s=float(b.Wall_s.mean()))
    for metric, label in (('Wall_s', 'Time_ratio'), ('Total_parameters', 'Parameter_ratio'), ('Regret', 'Regret_ratio')):
        x, y = a[metric].to_numpy(dtype=float), b[metric].to_numpy(dtype=float)
        ratios = y / x
        logs = np.log(ratios)
        center = logs.mean()
        half = student_t.ppf(.975, 4) * logs.std(ddof=1) / np.sqrt(5)
        ratio = np.exp([center, center - half, center + half]).tolist()
        row[label] = f'{ratio[0]:.3f} [{ratio[1]:.3f}, {ratio[2]:.3f}]'
    table.append(row)

In [7]:
summary_table = pd.DataFrame(table)
print(f"{hardware['gpu']} | {hardware['dtype']}")
display(summary_table.round(3))

Tesla T4 | float64


,Environment,BBP_time_s,Compressed_time_s,Time_ratio,Parameter_ratio,Regret_ratio
0,Weak,64.112,63.757,"0.995 [0.971, 1.018]","0.200 [0.191, 0.210]","0.988 [0.955, 1.021]"
1,Balanced,63.174,62.429,"0.988 [0.976, 1.000]","0.206 [0.198, 0.214]","0.986 [0.950, 1.022]"
2,KuaiRec,44.801,44.398,"0.991 [0.978, 1.004]","0.211 [0.202, 0.220]","1.002 [0.986, 1.019]"
